In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

#DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data/processed/bbh_32s_m1-xx_m2-xx_n100")
DATA_ROOT = Path("/home/victor/gw/cbc_pe/data/processed/bbh_32s_m1-xx_m2-xx_n100")

files = sorted(DATA_ROOT.glob("bbh_processed_32s_m1-*_m2-*_n100_*.h5"))

len(files), files[:3]

In [ ]:
# Extraer masa desde filename
def parse_masses_from_name(path):
    pattern = r"m1-(\d+)_m2-(\d+)"
    match = re.search(pattern, path.name)

    if match is None:
        return None, None

    return float(match.group(1)), float(match.group(2))

In [ ]:
# Resumen por archivo
rows = []

for path in files:
    expected_m1, expected_m2 = parse_masses_from_name(path)

    with h5py.File(path, "r") as f:
        X_shape = f["X"].shape
        y_shape = f["y"].shape

        m1 = f["parameters/mass_1"][:]
        m2 = f["parameters/mass_2"][:]
        spin1 = f["parameters/spin_1z"][:]
        spin2 = f["parameters/spin_2z"][:]
        chi_eff = f["parameters/chi_eff"][:]
        snr = f["snr/network"][:]
        distance = f["parameters/distance"][:]

        X_head = f["X"][:min(10, X_shape[0])]

        rows.append({
            "file": path.name,
            "path": str(path),
            "expected_m1": expected_m1,
            "expected_m2": expected_m2,
            "X_shape": X_shape,
            "y_shape": y_shape,

            "m1_min": float(m1.min()),
            "m1_max": float(m1.max()),
            "m2_min": float(m2.min()),
            "m2_max": float(m2.max()),

            "spin1_min": float(spin1.min()),
            "spin1_max": float(spin1.max()),
            "spin2_min": float(spin2.min()),
            "spin2_max": float(spin2.max()),
            "chi_eff_min": float(chi_eff.min()),
            "chi_eff_max": float(chi_eff.max()),

            "snr_min": float(snr.min()),
            "snr_max": float(snr.max()),
            "snr_mean": float(snr.mean()),
            "snr_std": float(snr.std()),

            "distance_min": float(distance.min()),
            "distance_max": float(distance.max()),
            "distance_mean": float(distance.mean()),

            "X_head_mean": float(X_head.mean()),
            "X_head_std": float(X_head.std()),
            "X_head_finite": bool(np.all(np.isfinite(X_head))),
        })

summary_df = pd.DataFrame(rows)
summary_df

In [ ]:
summary_df["mass_1_ok"] = (
    np.isclose(summary_df["m1_min"], summary_df["expected_m1"])
    & np.isclose(summary_df["m1_max"], summary_df["expected_m1"])
)

summary_df["mass_2_ok"] = (
    np.isclose(summary_df["m2_min"], summary_df["expected_m2"])
    & np.isclose(summary_df["m2_max"], summary_df["expected_m2"])
)

summary_df["spins_zero_ok"] = (
    np.isclose(summary_df["spin1_min"], 0.0)
    & np.isclose(summary_df["spin1_max"], 0.0)
    & np.isclose(summary_df["spin2_min"], 0.0)
    & np.isclose(summary_df["spin2_max"], 0.0)
)

summary_df["chi_eff_zero_ok"] = (
    np.isclose(summary_df["chi_eff_min"], 0.0)
    & np.isclose(summary_df["chi_eff_max"], 0.0)
)

summary_df["snr_ok"] = (
    (summary_df["snr_min"] >= 10.0)
    & (summary_df["snr_max"] <= 25.0)
)

summary_df["shape_ok"] = summary_df["X_shape"].apply(
    lambda s: s == (100, 3, 131072)
)

summary_df["finite_ok"] = summary_df["X_head_finite"]

check_cols = [
    "file",
    "mass_1_ok",
    "mass_2_ok",
    "spins_zero_ok",
    "chi_eff_zero_ok",
    "snr_ok",
    "shape_ok",
    "finite_ok",
]

summary_df[check_cols]

In [ ]:
for col in check_cols[1:]:
    print(col, ":", summary_df[col].sum(), "/", len(summary_df))

In [ ]:
summary_df.groupby("expected_m1")[["snr_mean", "snr_std", "distance_mean"]].describe()

In [ ]:
summary_df.groupby("expected_m1")[["snr_mean", "snr_std", "distance_mean"]].describe()

In [ ]:
sample_rows = []

for path in files:
    expected_m1, expected_m2 = parse_masses_from_name(path)

    with h5py.File(path, "r") as f:
        snr = f["snr/network"][:]
        distance = f["parameters/distance"][:]
        chirp_mass = f["parameters/chirp_mass"][:]
        total_mass = f["parameters/total_mass"][:]

        for i in range(len(snr)):
            sample_rows.append({
                "file": path.name,
                "mass_group": expected_m1,
                "sample_index": i,
                "snr": snr[i],
                "distance": distance[i],
                "chirp_mass": chirp_mass[i],
                "total_mass": total_mass[i],
            })

samples_df = pd.DataFrame(sample_rows)
samples_df.head()

In [ ]:
for col in ["snr", "distance"]:
    plt.figure(figsize=(8, 5))

    for mass_group, group in samples_df.groupby("mass_group"):
        plt.hist(group[col], bins=40, alpha=0.5, label=f"{mass_group:.0f}+{mass_group:.0f}")

    plt.xlabel(col)
    plt.ylabel("count")
    plt.title(f"{col} distribution by mass group")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
x_rows = []

for path in files:
    expected_m1, expected_m2 = parse_masses_from_name(path)

    with h5py.File(path, "r") as f:
        X = f["X"][:]

    x_mean = X.mean(axis=(1, 2))
    x_std = X.std(axis=(1, 2))
    x_rms = np.sqrt(np.mean(X**2, axis=(1, 2)))
    x_max_abs = np.max(np.abs(X), axis=(1, 2))

    for i in range(X.shape[0]):
        x_rows.append({
            "file": path.name,
            "mass_group": expected_m1,
            "sample_index": i,
            "x_mean": x_mean[i],
            "x_std": x_std[i],
            "x_rms": x_rms[i],
            "x_max_abs": x_max_abs[i],
        })

x_df = pd.DataFrame(x_rows)
x_df.head()

In [ ]:
for col in ["x_std", "x_rms", "x_max_abs"]:
    plt.figure(figsize=(8, 5))

    for mass_group, group in x_df.groupby("mass_group"):
        plt.hist(group[col], bins=40, alpha=0.5, label=f"{mass_group:.0f}+{mass_group:.0f}")

    plt.xlabel(col)
    plt.ylabel("count")
    plt.title(f"{col} distribution by mass group")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
clf_df = x_df.merge(
    samples_df[["file", "sample_index", "snr", "distance"]],
    on=["file", "sample_index"],
    how="left",
)

features = clf_df[["x_mean", "x_std", "x_rms", "x_max_abs", "snr"]].to_numpy()
labels = clf_df["mass_group"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.25,
    random_state=123,
    stratify=labels,
)

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, multi_class="auto"),
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)

print("accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

In [ ]:
features_no_snr = clf_df[["x_mean", "x_std", "x_rms", "x_max_abs"]].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    features_no_snr,
    labels,
    test_size=0.25,
    random_state=123,
    stratify=labels,
)

clf_no_snr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)

clf_no_snr.fit(X_train, y_train)
pred = clf_no_snr.predict(X_test)

print("accuracy without SNR:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

In [ ]:
unique_files = clf_df[["file", "mass_group"]].drop_duplicates()

train_files = []
test_files = []

rng = np.random.default_rng(123)

for mass_group, group in unique_files.groupby("mass_group"):
    files_group = group["file"].to_numpy()
    rng.shuffle(files_group)

    test_files.extend(files_group[:1])
    train_files.extend(files_group[1:])

train_files = set(train_files)
test_files = set(test_files)

train_mask = clf_df["file"].isin(train_files)
test_mask = clf_df["file"].isin(test_files)

X_train = clf_df.loc[train_mask, ["x_mean", "x_std", "x_rms", "x_max_abs"]].to_numpy()
y_train = clf_df.loc[train_mask, "mass_group"].to_numpy()

X_test = clf_df.loc[test_mask, ["x_mean", "x_std", "x_rms", "x_max_abs"]].to_numpy()
y_test = clf_df.loc[test_mask, "mass_group"].to_numpy()

clf_file_split = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)

clf_file_split.fit(X_train, y_train)
pred = clf_file_split.predict(X_test)

print("Train files:", sorted(train_files))
print("Test files:", sorted(test_files))
print("accuracy file split:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
X_rows = []
y_rows = []
file_rows = []

stride = 64  # 131072 / 64 = 2048 samples per detector

for path in files:
    expected_m1, expected_m2 = parse_masses_from_name(path)

    with h5py.File(path, "r") as f:
        X = f["X"][:, :, ::stride]  # (100, 3, 2048)

    X_flat = X.reshape(X.shape[0], -1)

    X_rows.append(X_flat)
    y_rows.extend([expected_m1] * X.shape[0])
    file_rows.extend([path.name] * X.shape[0])

X_down = np.concatenate(X_rows, axis=0)
y_mass = np.asarray(y_rows)
file_arr = np.asarray(file_rows)

print("X_down:", X_down.shape)
print("y_mass:", y_mass.shape)

In [ ]:
unique_files = pd.DataFrame({
    "file": file_arr,
    "mass_group": y_mass,
}).drop_duplicates()

train_files = []
test_files = []

rng = np.random.default_rng(123)

for mass_group, group in unique_files.groupby("mass_group"):
    fg = group["file"].to_numpy()
    rng.shuffle(fg)
    test_files.extend(fg[:1])
    train_files.extend(fg[1:])

train_mask = np.isin(file_arr, train_files)
test_mask = np.isin(file_arr, test_files)

X_train = X_down[train_mask]
X_test = X_down[test_mask]
y_train = y_mass[train_mask]
y_test = y_mass[test_mask]

clf_raw = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, C=1.0),
)

clf_raw.fit(X_train, y_train)
pred = clf_raw.predict(X_test)

print("accuracy downsampled raw X:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

In [ ]:
def windowed_rms_features(X, n_windows=64):
    """
    X shape: (n_samples, n_detectors, n_time)
    returns shape: (n_samples, n_detectors * n_windows)
    """
    n, c, t = X.shape
    usable = (t // n_windows) * n_windows
    X = X[:, :, :usable]
    Xw = X.reshape(n, c, n_windows, usable // n_windows)
    rms = np.sqrt(np.mean(Xw**2, axis=-1))
    return rms.reshape(n, c * n_windows)

In [ ]:
X_feat_rows = []
y_rows = []
file_rows = []

for path in files:
    expected_m1, expected_m2 = parse_masses_from_name(path)

    with h5py.File(path, "r") as f:
        X = f["X"][:]

    feat = windowed_rms_features(X, n_windows=64)

    X_feat_rows.append(feat)
    y_rows.extend([expected_m1] * X.shape[0])
    file_rows.extend([path.name] * X.shape[0])

X_winrms = np.concatenate(X_feat_rows, axis=0)
y_mass = np.asarray(y_rows)
file_arr = np.asarray(file_rows)

print(X_winrms.shape)

In [ ]:
unique_files = pd.DataFrame({
    "file": file_arr,
    "mass_group": y_mass,
}).drop_duplicates()

train_files = []
test_files = []

rng = np.random.default_rng(123)

for mass_group, group in unique_files.groupby("mass_group"):
    fg = group["file"].to_numpy()
    rng.shuffle(fg)
    test_files.extend(fg[:1])
    train_files.extend(fg[1:])

train_mask = np.isin(file_arr, train_files)
test_mask = np.isin(file_arr, test_files)

X_train = X_winrms[train_mask]
X_test = X_winrms[test_mask]
y_train = y_mass[train_mask]
y_test = y_mass[test_mask]

clf_winrms = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, C=1.0),
)

clf_winrms.fit(X_train, y_train)
pred = clf_winrms.predict(X_test)

print("accuracy windowed RMS features:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

In [ ]:
def plot_spectrogram_sample(path, sample_idx=0, detector_idx=0):
    with h5py.File(path, "r") as f:
        x = f["X"][sample_idx, detector_idx]
        fs = float(f.attrs["sampling_frequency"])

    plt.figure(figsize=(10, 5))
    plt.specgram(x, NFFT=1024, Fs=fs, noverlap=768)
    plt.ylim(0, 512)
    plt.xlabel("Time [s]")
    plt.ylabel("Frequency [Hz]")
    plt.title(f"{path.name} | sample {sample_idx} | detector {detector_idx}")
    plt.colorbar(label="Power")
    plt.tight_layout()
    plt.show()

In [ ]:
for mass in [20, 40, 60, 80]:
    path = [p for p in files if f"m1-{mass}_m2-{mass}" in p.name][0]
    plot_spectrogram_sample(path, sample_idx=0, detector_idx=0)

## 8. Temporal feature audit

The previous checks showed that global features such as mean, standard deviation, RMS and maximum absolute value do not separate the mass groups well.

This does not prove that the simulations are wrong. It only shows that global features are too weak.

In this section we test whether more time-local representations contain useful information:

1. RMS in temporal windows;
2. crop around the strongest network response;
3. downsampled crop classification;
4. spectrograms around the strongest response.

In [ ]:
def make_file_level_split(file_arr, y_mass, test_files_per_class=1, seed=123):
    """
    Create a train/test split where whole files are assigned either to train or test.

    This avoids mixing samples from the same HDF5 file across train and test.
    """
    unique_files = pd.DataFrame({
        "file": file_arr,
        "mass_group": y_mass,
    }).drop_duplicates()

    train_files = []
    test_files = []

    rng = np.random.default_rng(seed)

    for mass_group, group in unique_files.groupby("mass_group"):
        files_group = group["file"].to_numpy()
        rng.shuffle(files_group)

        test_files.extend(files_group[:test_files_per_class])
        train_files.extend(files_group[test_files_per_class:])

    train_files = set(train_files)
    test_files = set(test_files)

    train_mask = np.isin(file_arr, list(train_files))
    test_mask = np.isin(file_arr, list(test_files))

    return train_mask, test_mask, train_files, test_files

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def evaluate_mass_classifier(X_feat, y_mass, file_arr, title="", seed=123):
    """
    Train and evaluate a simple classifier using a file-level split.
    """
    train_mask, test_mask, train_files, test_files = make_file_level_split(
        file_arr=file_arr,
        y_mass=y_mass,
        test_files_per_class=1,
        seed=seed,
    )

    X_train = X_feat[train_mask]
    X_test = X_feat[test_mask]
    y_train = y_mass[train_mask]
    y_test = y_mass[test_mask]

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, C=1.0),
    )

    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    print("=" * 80)
    print(title)
    print("=" * 80)
    print("Train files:", sorted(train_files))
    print("Test files:", sorted(test_files))
    print()
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print()
    print("accuracy:", accuracy_score(y_test, pred))
    print()
    print(classification_report(y_test, pred))
    print()
    print("confusion matrix:")
    print(confusion_matrix(y_test, pred))

    return clf, pred, y_test

Here
- Training with 4 files per mass
- Testing with 1 file per mass


In [ ]:
def windowed_rms_features(X, n_windows=64):
    """
    Compute RMS in temporal windows.

    Parameters
    ----------
    X : np.ndarray
        Shape (n_samples, n_detectors, n_time).
    n_windows : int
        Number of temporal windows.

    Returns
    -------
    features : np.ndarray
        Shape (n_samples, n_detectors * n_windows).
    """
    n, c, t = X.shape

    usable = (t // n_windows) * n_windows
    X = X[:, :, :usable]

    Xw = X.reshape(n, c, n_windows, usable // n_windows)

    rms = np.sqrt(np.mean(Xw**2, axis=-1))

    return rms.reshape(n, c * n_windows)

In [ ]:
X_feat_rows = []
y_rows = []
file_rows = []

for path in files:
    expected_m1, expected_m2 = parse_masses_from_name(path)

    with h5py.File(path, "r") as f:
        X = f["X"][:]

    feat = windowed_rms_features(X, n_windows=64)

    X_feat_rows.append(feat)
    y_rows.extend([expected_m1] * X.shape[0])
    file_rows.extend([path.name] * X.shape[0])

X_winrms = np.concatenate(X_feat_rows, axis=0)
y_mass = np.asarray(y_rows)
file_arr = np.asarray(file_rows)

print("X_winrms:", X_winrms.shape)
print("y_mass:", y_mass.shape)
print("files:", len(np.unique(file_arr)))

In [ ]:
clf_winrms, pred_winrms, y_test_winrms = evaluate_mass_classifier(
    X_feat=X_winrms,
    y_mass=y_mass,
    file_arr=file_arr,
    title="Mass classification using windowed RMS features",
)

Cómo interpretar este paso

Si accuracy sigue cerca de 0.25–0.30, entonces la energía por ventanas tampoco separa bien las masas.

Si sube claramente, por ejemplo 0.50+, entonces hay información temporal básica, pero tus features globales la estaban borrando.

Crop alrededor del máximo de la red

Esta es la prueba importante. En 32 s, la señal puede estar diluida. Vamos a recortar una ventana de 4 s alrededor del máximo de energía de red.

In [ ]:
def crop_around_network_peak(X, crop_seconds=4.0, fs=4096.0):
    """
    Crop each sample around the maximum network amplitude.

    This is an approximate way to localize the most informative region without
    using injection metadata.

    Parameters
    ----------
    X : np.ndarray
        Shape (n_samples, n_detectors, n_time).
    crop_seconds : float
        Crop duration in seconds.
    fs : float
        Sampling frequency.

    Returns
    -------
    crops : np.ndarray
        Shape (n_samples, n_detectors, crop_len).
    peak_indices : np.ndarray
        Peak index used for each sample.
    """
    n, c, t = X.shape
    crop_len = int(round(crop_seconds * fs))

    if crop_len > t:
        raise ValueError("crop_len cannot be larger than signal length.")

    crops = np.zeros((n, c, crop_len), dtype=X.dtype)

    network_abs = np.sqrt(np.sum(X**2, axis=1))  # shape (n_samples, n_time)
    peak_indices = np.argmax(network_abs, axis=1)

    for i in range(n):
        center = int(peak_indices[i])

        start = center - crop_len // 2
        end = start + crop_len

        if start < 0:
            start = 0
            end = crop_len

        if end > t:
            end = t
            start = t - crop_len

        crops[i] = X[i, :, start:end]

    return crops, peak_indices

In [ ]:
X_crop_rows = []
y_rows = []
file_rows = []
peak_rows = []

crop_seconds = 4.0
downsample_stride = 8

for path in files:
    expected_m1, expected_m2 = parse_masses_from_name(path)

    with h5py.File(path, "r") as f:
        X = f["X"][:]
        fs = float(f.attrs.get("sampling_frequency", 4096.0))

    X_crop, peak_indices = crop_around_network_peak(
        X,
        crop_seconds=crop_seconds,
        fs=fs,
    )

    # Downsample the crop to keep the classifier cheap.
    X_crop_down = X_crop[:, :, ::downsample_stride]
    X_flat = X_crop_down.reshape(X_crop_down.shape[0], -1)

    X_crop_rows.append(X_flat)
    y_rows.extend([expected_m1] * X.shape[0])
    file_rows.extend([path.name] * X.shape[0])
    peak_rows.extend(peak_indices.tolist())

X_crop_down = np.concatenate(X_crop_rows, axis=0)
y_mass_crop = np.asarray(y_rows)
file_arr_crop = np.asarray(file_rows)
peak_indices_all = np.asarray(peak_rows)

print("X_crop_down:", X_crop_down.shape)
print("y_mass_crop:", y_mass_crop.shape)
print("peak_indices_all:", peak_indices_all.shape)

In [ ]:
clf_crop, pred_crop, y_test_crop = evaluate_mass_classifier(
    X_feat=X_crop_down,
    y_mass=y_mass_crop,
    file_arr=file_arr_crop,
    title="Mass classification using 4 s crop around network peak",
)

Posibles resultados:

accuracy ~0.25-0.30:
    ni siquiera el crop local contiene información accesible para este clasificador lineal.

accuracy ~0.40-0.60:
    sí hay información temporal local, pero estaba diluida en los 32 s.

accuracy >0.70:
    la información está ahí; el problema principal era localización/dilución temporal.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(peak_indices_all / 4096.0, bins=50)
plt.xlabel("Peak time inside 32 s segment [s]")
plt.ylabel("count")
plt.title("Distribution of network peak time")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Si el peak está muy distribuido, los modelos no invariantes a traslación sufren.
Si está siempre cerca del final o en una zona concreta, la localización no es el principal problema.

In [ ]:
def plot_spectrogram_crop_around_peak(
    path,
    sample_idx=0,
    detector_idx=0,
    crop_seconds=4.0,
    nfft=512,
    noverlap=384,
    fmax=512,
):
    with h5py.File(path, "r") as f:
        X = f["X"][sample_idx]
        fs = float(f.attrs.get("sampling_frequency", 4096.0))

    network_abs = np.sqrt(np.sum(X**2, axis=0))
    peak_idx = int(np.argmax(network_abs))

    crop_len = int(round(crop_seconds * fs))
    start = peak_idx - crop_len // 2
    end = start + crop_len

    if start < 0:
        start = 0
        end = crop_len

    if end > X.shape[-1]:
        end = X.shape[-1]
        start = end - crop_len

    x = X[detector_idx, start:end]

    plt.figure(figsize=(10, 5))
    plt.specgram(x, NFFT=nfft, Fs=fs, noverlap=noverlap)
    plt.ylim(0, fmax)
    plt.xlabel("Time inside crop [s]")
    plt.ylabel("Frequency [Hz]")
    plt.title(
        f"{path.name} | sample {sample_idx} | detector {detector_idx} | crop around peak"
    )
    plt.colorbar(label="Power")
    plt.tight_layout()
    plt.show()

In [ ]:
for mass in [20, 40, 60, 80]:
    mass_files = [p for p in files if f"m1-{mass}_m2-{mass}" in p.name]
    path = mass_files[0]

    print(path.name)
    plot_spectrogram_crop_around_peak(
        path,
        sample_idx=0,
        detector_idx=0,
        crop_seconds=4.0,
    )

20+20 debería tender a tener chirp más largo.
80+80 debería ser más compacto.

In [ ]:
def build_crop_downsampled_collection(files, crop_seconds=4.0, downsample_stride=8):
    X_rows = []
    y_rows = []
    file_rows = []
    peak_rows = []

    for path in files:
        expected_m1, expected_m2 = parse_masses_from_name(path)

        with h5py.File(path, "r") as f:
            X = f["X"][:]
            fs = float(f.attrs.get("sampling_frequency", 4096.0))

        X_crop, peak_indices = crop_around_network_peak(
            X,
            crop_seconds=crop_seconds,
            fs=fs,
        )

        X_crop_down = X_crop[:, :, ::downsample_stride]
        X_flat = X_crop_down.reshape(X_crop_down.shape[0], -1)

        X_rows.append(X_flat)
        y_rows.extend([expected_m1] * X.shape[0])
        file_rows.extend([path.name] * X.shape[0])
        peak_rows.extend(peak_indices.tolist())

    return (
        np.concatenate(X_rows, axis=0),
        np.asarray(y_rows),
        np.asarray(file_rows),
        np.asarray(peak_rows),
    )

In [ ]:
for crop_seconds in [2.0, 4.0, 8.0]:
    X_crop_feat, y_crop, file_crop, peak_crop = build_crop_downsampled_collection(
        files,
        crop_seconds=crop_seconds,
        downsample_stride=8,
    )

    evaluate_mass_classifier(
        X_feat=X_crop_feat,
        y_mass=y_crop,
        file_arr=file_crop,
        title=f"Mass classification using {crop_seconds:.0f} s crop around network peak",
    )

Si 2s va mejor: la información está muy concentrada cerca del merger.
Si 8s va mejor: necesitas algo más de inspiral.
Si ninguno mejora: problema de ruido/procesado/modelo.

In [ ]:
peak_df = pd.DataFrame({
    "mass_group": y_mass_crop,
    "file": file_arr_crop,
    "peak_time": peak_indices_all / 4096.0,
})

for mass_group, group in peak_df.groupby("mass_group"):
    plt.figure(figsize=(7, 4))
    plt.hist(group["peak_time"], bins=40)
    plt.xlabel("Peak time inside 32 s segment [s]")
    plt.ylabel("count")
    plt.title(f"Peak time distribution — {mass_group:.0f}+{mass_group:.0f}")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()